# Analysis 01
Chunking the Samples based on timepoints

Load Files

In [ ]:
# Navigate to data directory
import os
import pandas as pd
import warnings
import winsound

# Declare filenames
ROOT_DATA_DIR = (r"~\Data\Eyetracking_01_Preprocessed_Data") # folder where the Input data sits
DATA_DIR_OUTPUT =  (r"~\Data\Eyetracking_02_Data_Trials")

# Suppress the SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# Iterate through subdirectories
for subdir in os.listdir(ROOT_DATA_DIR):
    # Set the path to the directory for this subdirectory
    subdir_path = os.path.join(ROOT_DATA_DIR, subdir)
    print('Subdirectory:', subdir_path)

    # Check if the subdir_path is a directory and does not end with '_02' # TO DO: Add Retrieval files. And extraction code
    if os.path.isdir(subdir_path) and not subdir.endswith('_02'):
        print('Subdirectory:', subdir_path)

        # Scan the subdirectory for files
        file_list = os.listdir(subdir_path)

        # Check if the required files exist in the subdirectory
        if all(file_name.endswith('.xlsx') or file_name.endswith('.csv') for file_name in file_list):
            
            # Get the paths to the required files
            movie_file_path = os.path.join(subdir_path, [file_name for file_name in file_list if file_name.endswith('Movie_Timestamps.xlsx')][0])
            samples_file_path = os.path.join(subdir_path, [file_name for file_name in file_list if file_name.endswith('Samples.csv')][0])
            fixation_file_path = os.path.join(subdir_path, [file_name for file_name in file_list if file_name.endswith('Fixation.xlsx')][0])
            saccade_file_path = os.path.join(subdir_path, [file_name for file_name in file_list if file_name.endswith('Saccade.xlsx')][0])


            # ===== LOAD THE FILE BASED ON STRINGS ===== #

            # Load the files containing the start and end time points
            DF_ts_movie = pd.read_excel((movie_file_path), index_col=0)
            movie_num = DF_ts_movie['movie_num'].loc[0]
            print('Timestamps loaded ...')

            # Load the data file from which to extract rows
            DF_SAMPLES =  pd.read_csv((samples_file_path), index_col=0)
            DF_FIXATION = pd.read_excel((fixation_file_path), index_col=0)
            DF_SACCADES = pd.read_excel((saccade_file_path), index_col=0)
            # DF_TRIALS =   pd.read_excel((trial_Str_path), index_col=0)

            # Create new  Output folder if folder does not already exist: 
            folder_name = subdir
            
            DATA_OUTPUT = os.path.join(DATA_DIR_OUTPUT, folder_name)
            if not os.path.exists(DATA_OUTPUT):
                os.mkdir(DATA_OUTPUT)

            # Assign the indexing points
            start_time = DF_ts_movie['t_start(ms)']
            end_time = DF_ts_movie['t_end(ms)']
            ORDER = 1 # of 6
            BLOCK = 1   # of 3

            # Get file prefix from a filename, for naming later on
            NUM_FileStart =  subdir
            print(NUM_FileStart)

            # Iterate over each row in the timepoints file
            # Loop over the 3 blocks
            for BLOCK in range(1, 4):
                print('BLOCK', BLOCK)
                # Loop over the 6 sessions per block
                for ORDER in range(1, 7):
                    print('ORDER', ORDER)
                    row = (BLOCK - 1) * 6 + (ORDER - 1)
                    movie_num = DF_ts_movie['movie_num'].loc[row]

                    # Extract Samples Data Chunk
                    extracted_data_samples = DF_SAMPLES[(DF_SAMPLES['tSample'] >= start_time[row]) & 
                                                        (DF_SAMPLES['tSample'] <= end_time[row])]
                    extracted_data_samples['Movie'] = movie_num

                    # Extract Fixations Data Chunk
                    extracted_data_fixations = DF_FIXATION[(DF_FIXATION['tStart'] >= start_time[row]) & 
                                                           (DF_FIXATION['tEnd'] <= end_time[row])]
                    extracted_data_fixations['Movie'] = movie_num

                    # Extract Saccades Data Chunk
                    extracted_data_saccades = DF_SACCADES[(DF_SACCADES['tStart'] >= start_time[row]) & 
                                                          (DF_SACCADES['tEnd'] <= end_time[row])]
                    extracted_data_saccades['Movie'] = movie_num
                    
                    



                    
                    # ===== SAVE THE DATA CHUNKS ===== #
                    # Make master list of dataframes to write
                    allDataFrames = [extracted_data_samples, extracted_data_fixations, extracted_data_saccades] # the dataframes
                    allNames = ['Samples', 'Fixation','Saccade'] # what they're called
                    
                    outFilename = '%s\%s_Block_%s_Order_%s_%s.xlsx' % (DATA_OUTPUT, NUM_FileStart, BLOCK, ORDER, movie_num)
                    
                    # Check if the output file already exists
                    if not os.path.exists(outFilename):
                        # Output file does not exist, save the dataframes
                        for i in range(len(allNames)):
                            # Generate the output file name for the specific dataframe
                            specificOutFilename = '%s\%s_%s_Block_%s_Order_%s_%s.xlsx' % (DATA_OUTPUT, NUM_FileStart, allNames[i], BLOCK, ORDER, movie_num)
                            
                            # Check if the specific output file already exists
                            if not os.path.exists(specificOutFilename):
                                # Specific output file does not exist, save the dataframe to Excel
                                print('Saving %s output as %s...' % (allNames[i], specificOutFilename))
                                allDataFrames[i].to_excel(specificOutFilename, float_format='%.1f', index=False, engine='openpyxl')
                            else:
                                print('Skipping %s output, file already exists: %s' % (allNames[i], specificOutFilename))
                    else:
                        print('Skipping block %s, order %s, files already exist.' % (BLOCK, ORDER))
               

                    # # Write dataframes to .xlsx files
                    # for i in range(len(allNames)):
                    #     outFilename = '%s\%s_%s_Block %s_Order_%s_%s.xlsx'%(DATA_OUTPUT, NUM_FileStart, allNames[i], BLOCK, ORDER, movie_num)
                    #     print('   Saving %s output as %s...'%(allNames[i],outFilename))
                    #     allDataFrames[i].to_excel(outFilename,float_format='%.1f',index=False, engine='openpyxl')

            else:
                print('Processing subdirectory %s complete'%(subdir))

        else:
            print('Required files not found in subdirectory.')

    # Move to the next subdirectory    
    # Check if the current subdirectory is the last one in the list
    if subdir == os.listdir(ROOT_DATA_DIR)[-1]:
        print('Processed all subdirectories.')
        winsound.MessageBeep()
        break

In [2]:
# # Navigate to data directory

# import os
# import pandas as pd

# # Declare filenames
# ROOT_DATA_DIR = (r"~\Data\Eyetracking_01_Preprocessed_Data\001") # folder where the Input data sits
# DATA_DIR_OUTPUT =  (r"~\Data\Eyetracking_02_Data_Trials")

# # Navigate to data directory
# os.chdir(ROOT_DATA_DIR)


# # Get file name based on "Movie_Timestamps.xlsx" (not participant specific)
# movie_file = [file_name for file_name in os.listdir('.') if file_name.endswith("Movie_Timestamps.xlsx")]
# movie_Str = ' '.join(map(str, movie_file))
# print('Movie_Timestamps file found ...')
# print(movie_Str)

# # Get file prefix from filename, for naming later on
# NUM_FileStart =  movie_Str[:6]
# # print(NUM_FileStart)

# # Get File of Samples.csv
# samples_file = [file_name for file_name in os.listdir('.') if file_name.startswith(NUM_FileStart) and file_name.endswith("Samples.csv")]
# samples_Str = ' '.join(map(str, samples_file))
# print('Samples loaded ...')

# # Load the file containing the start and end time points
# DF_ts_movie = pd.read_excel((movie_Str), index_col=0)
# movie_num = DF_ts_movie['movie_num'].loc[0]
# print('Timestamps loaded ...')
# # print(movie_num)
# # print(DF_ts_movie.head())

# # Load the data file from which to extract rows
# DF_Samples = pd.read_csv((samples_Str), index_col=0)
# # print(DF_Samples.head())


# # Create new  Output folder if folder does not already exist: 
# folder_name = movie_Str[:3]

# DATA_OUTPUT = os.path.join(DATA_DIR_OUTPUT, folder_name)
# if not os.path.exists(DATA_OUTPUT):
#     os.mkdir(DATA_OUTPUT)

# # Assign the indexing points
# start_time = DF_ts_movie['t_start(ms)']
# end_time = DF_ts_movie['t_end(ms)']
# ORDER = 1 # of 6
# BLOCK = 1   # of 3


# # Iterate over each row in the timepoints file
# # Loop over the 3 blocks
# for BLOCK in range(1, 4):
#     print(BLOCK)
#     # Loop over the 6 sessions per block
#     for ORDER in range(1, 7):
#         print(ORDER)
#         row = (BLOCK - 1) * 6 + (ORDER - 1)
        
#         # Extract Data Chunk
#         extracted_data = DF_Samples[(DF_Samples['tSample'] >= start_time[row]) & 
#                                     (DF_Samples['tSample'] <= end_time[row])]
#         movie_num = DF_ts_movie['movie_num'].loc[row]
#         extracted_data['Movie'] = movie_num

#         # Name File and save output
#         OUT_FILE_Name = '%s\%s_Block %s_ORDER_%s_%s.xlsx'%(DATA_OUTPUT, NUM_FileStart, BLOCK, ORDER, movie_num)
#         extracted_data.to_excel(OUT_FILE_Name)
                

